# Part C: Nematic Order Analysis — *P. aeruginosa* Mechanotaxis
## Publication Reference
Based on: **Kühn et al. (2021)** *PNAS* 2101759118 and **Basaran et al. (2022)** *eLife* 72187

## Scientific Context
This notebook quantifies the collective orientational order of *P. aeruginosa* colonies by computing
the **Nematic Order Parameter (Sr)** from TrackMate-extracted single-cell tracking data.

Two complementary formulations are implemented throughout:
- **Eq. 1 — Legendre P₂:** `Sr₁ = ⟨(3cos²θ − 1)/2⟩` — absolute orientational order
- **Eq. 2 — Basaran radial:** `Sr₂ = ⟨cos(2(θᵢ − φᵢ))⟩` — alignment relative to colony expansion direction (Basaran 2022)

## Analysis Sections
1. **C.1** — Global Sr for all frames, all 9 conditions
2. **C.2** — Quadrant-based Sr (FOV divided into 4 × 66.5 µm regions)
3. **C.3** — Distance-based local Sr (30 µm neighbourhood radius)
4. **C.4** — Per-cell Sr from a randomly selected representative frame
5. **C.5** — Sr vs neighbourhood radius sweep (0–133 µm) to characterise length-scale dependence

## Input
TrackMate CSV exports (one per condition) in `data/trackmate/`.
Expected columns: `POSITION_X`, `POSITION_Y`, `POSITION_T`, `ELLIPSE_THETA` (all in µm / s / radians).

## Conditions
9 conditions: WT / pilH / pilGcpdA × Dense / Medium / Dilute


## Section 0 — Imports & Configuration
Update `DATA_DIR` and `OUTPUT_DIR` if your folder structure differs.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# =============================================================================
# USER SETTINGS — edit these before running
# =============================================================================
DATA_DIR   = os.path.join(os.getcwd(), "data", "trackmate")   # TrackMate CSV files
OUTPUT_DIR = os.path.join(os.getcwd(), "data", "results")     # Output CSVs and figures

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Imaging / analysis parameters
PIXEL_SIZE    = 0.065   # µm/pixel
FRAME_INT     = 5       # seconds between frames
FOV_SIZE      = 133.0   # µm (square field of view)
HALF_FOV      = FOV_SIZE / 2

# Distance-filter parameters
MAX_DISTANCE  = 20      # µm — neighbourhood radius for C.3 & C.4 (determined by FIJI line
                         #        measurement of average collective swirl radius; see report s3.1.2)
MAX_DIST_CELL = 20      # µm — per-cell neighbourhood radius (same as MAX_DISTANCE;
                         #        Sr validated as time-independent so one frame suffices)

# Distance sweep parameters (Section C.5)
DIST_SWEEP    = np.linspace(2, FOV_SIZE, num=67)  # 2 µm to 133 µm in 67 steps

# Reproducibility
RANDOM_SEED   = 42

file_names = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])
print(f"Found {len(file_names)} condition file(s) in {DATA_DIR}:")
for f in file_names:
    print(f"  {f}")


## Section 1 — Helper Functions
All analysis functions defined once here and reused throughout.


In [ ]:
# =============================================================================
# 1.1  Sr formulas
# =============================================================================

def sr_legendre(theta):
    """
    Eq. 1 — Legendre P₂ nematic order (absolute orientational order).

    Sr₁ = ⟨(3cos²θ − 1) / 2⟩

    :param theta: 1-D array of cell orientation angles (radians, from ellipse fit)
    :return: float Sr₁ ∈ [−0.5, 1.0]
    """
    return float(np.mean((3 * np.cos(theta)**2 - 1) / 2))


def sr_basaran(theta, phi):
    """
    Eq. 2 — Basaran radial nematic order (alignment relative to colony centre).
    Reference: Basaran et al. (2022) eLife 72187.

    Sr₂ = ⟨cos(2(θᵢ − φᵢ))⟩

    :param theta: 1-D array of cell orientation angles (radians)
    :param phi:   1-D array of polar angles of each cell relative to colony centroid (radians)
    :return: float Sr₂ ∈ [−1.0, 1.0]
    """
    return float(np.mean(np.cos(2 * (theta - phi))))


# =============================================================================
# 1.2  Colony geometry
# =============================================================================

def compute_phi(x, y, cx, cy):
    """
    Angular position φᵢ of each cell relative to a reference centre (cx, cy).

    :param x, y:  Arrays of cell coordinates (µm)
    :param cx, cy: Reference centre coordinates (µm)
    :return: Array of φ values in radians, range (−π, π]
    """
    return np.arctan2(y - cy, x - cx)


# =============================================================================
# 1.3  Data loading and cleaning
# =============================================================================

REQUIRED_COLS = ['POSITION_X', 'POSITION_Y', 'POSITION_T', 'ELLIPSE_THETA']

def load_trackmate_csv(filepath):
    """
    Load a TrackMate CSV export, coerce numeric columns, and drop invalid rows.
    TrackMate exports often contain non-numeric header rows — low_memory=False
    prevents mixed-type inference warnings.

    :param filepath: Path to TrackMate CSV file
    :return: Cleaned DataFrame with REQUIRED_COLS as float64
    """
    data = pd.read_csv(filepath, low_memory=False)
    for col in REQUIRED_COLS:
        data[col] = pd.to_numeric(data[col], errors='coerce')
    data = data.dropna(subset=REQUIRED_COLS).reset_index(drop=True)
    return data


# =============================================================================
# 1.4  Quadrant filter
# =============================================================================

def filter_by_quadrant(group, quadrant, half_fov):
    """
    Return cells within a given quadrant of the field of view.
    Origin (0, 0) is bottom-left as per TrackMate coordinate convention.

    Quadrant layout:
        Q1: top-left     Q2: top-right
        Q3: bottom-left  Q4: bottom-right

    :param group:    DataFrame with POSITION_X and POSITION_Y columns
    :param quadrant: 'Q1', 'Q2', 'Q3', or 'Q4'
    :param half_fov: Half the FOV width/height in µm
    :return: Filtered DataFrame (empty if quadrant string invalid)
    """
    x, y = group['POSITION_X'], group['POSITION_Y']
    if quadrant == 'Q1':   return group[(x <  half_fov) & (y >= half_fov)]
    elif quadrant == 'Q2': return group[(x >= half_fov) & (y >= half_fov)]
    elif quadrant == 'Q3': return group[(x <  half_fov) & (y <  half_fov)]
    elif quadrant == 'Q4': return group[(x >= half_fov) & (y <  half_fov)]
    else:                  return pd.DataFrame()

# Geometric centres for each quadrant (used as φ reference)
QUADRANT_CENTRES = {
    'Q1': (HALF_FOV / 2,         3 * HALF_FOV / 2),
    'Q2': (3 * HALF_FOV / 2,     3 * HALF_FOV / 2),
    'Q3': (HALF_FOV / 2,         HALF_FOV / 2),
    'Q4': (3 * HALF_FOV / 2,     HALF_FOV / 2),
}


# =============================================================================
# 1.5  Distance-based neighbour filter
# =============================================================================

def get_neighbour_mask(x, y, i, max_dist):
    """
    Boolean mask of cells within max_dist µm of cell i (self excluded).

    :param x, y:    Coordinate arrays (µm)
    :param i:       Index of the reference cell
    :param max_dist: Radius threshold (µm)
    :return: Boolean array of length len(x)
    """
    d = np.sqrt((x - x[i])**2 + (y - y[i])**2)
    return (d <= max_dist) & (d > 0)


def sr_legendre_local(theta, x, y, max_dist):
    """
    Mean Eq. 1 Sr over non-overlapping local neighbourhoods.

    Each cell i is processed once; its neighbours within max_dist are used to
    compute a local Sr₁. Cells already used as neighbours are skipped to avoid
    double-counting. The frame-level Sr is the weighted mean across groups.

    :return: float (weighted mean Sr₁) or np.nan if no valid groups found
    """
    num_cells = len(x)
    processed = set()
    sr_sum, count = 0.0, 0

    for i in range(num_cells):
        if i in processed:
            continue
        mask = get_neighbour_mask(x, y, i, max_dist)
        nb_theta = theta[mask]
        if len(nb_theta) == 0:
            continue
        sr_sum += sr_legendre(nb_theta) * len(nb_theta)
        count  += len(nb_theta)
        processed.update(np.where(mask)[0])

    return sr_sum / count if count > 0 else np.nan


def sr_basaran_local(theta, x, y, max_dist):
    """
    Mean Eq. 2 Sr over non-overlapping local neighbourhoods.

    φ is computed dynamically from the centroid of each neighbourhood group,
    capturing local radial alignment rather than global colony direction.

    :return: float (weighted mean Sr₂) or np.nan if no valid groups found
    """
    num_cells = len(x)
    processed = set()
    sr_sum, count = 0.0, 0

    for i in range(num_cells):
        if i in processed:
            continue
        mask = get_neighbour_mask(x, y, i, max_dist)
        nb_theta = theta[mask]
        nb_x     = x[mask]
        nb_y     = y[mask]
        if len(nb_theta) == 0:
            continue
        # φ relative to local neighbourhood centroid
        cx, cy   = np.mean(nb_x), np.mean(nb_y)
        nb_phi   = compute_phi(nb_x, nb_y, cx, cy)
        sr_sum  += sr_basaran(nb_theta, nb_phi) * len(nb_theta)
        count   += len(nb_theta)
        processed.update(np.where(mask)[0])

    return sr_sum / count if count > 0 else np.nan


# =============================================================================
# 1.6  Per-cell Sr (returns spatial map for one frame)
# =============================================================================

def sr_per_cell_legendre(theta, x, y, max_dist):
    """
    Eq. 1 Sr for each non-overlapping cell neighbourhood.
    Returns a list of (x_i, y_i, cell_index, Sr₁) tuples.
    """
    num_cells = len(x)
    processed = set()
    results   = []
    for i in range(num_cells):
        if i in processed:
            continue
        mask   = get_neighbour_mask(x, y, i, max_dist)
        nb_th  = theta[mask]
        if len(nb_th) == 0:
            continue
        results.append((x[i], y[i], i, sr_legendre(nb_th)))
        processed.update(np.where(mask)[0])
    return results


def sr_per_cell_basaran(theta, x, y, max_dist):
    """
    Eq. 2 Sr for each non-overlapping cell neighbourhood.
    Returns a list of (x_i, y_i, cell_index, Sr₂) tuples.

    BUG FIX (02.03.2026): indentation error in calculate_nematic_order_for_each_cell2
    caused center_x/y computation to fall outside the for loop body, meaning all cells
    shared the centroid of the last processed group. Fixed here by correct indentation.
    """
    num_cells = len(x)
    processed = set()
    results   = []
    for i in range(num_cells):
        if i in processed:
            continue
        mask   = get_neighbour_mask(x, y, i, max_dist)
        nb_th  = theta[mask]
        nb_x   = x[mask]
        nb_y   = y[mask]
        if len(nb_th) == 0:
            continue
        cx, cy   = np.mean(nb_x), np.mean(nb_y)    # local centroid — MUST be inside loop
        nb_phi   = compute_phi(nb_x, nb_y, cx, cy)
        results.append((x[i], y[i], i, sr_basaran(nb_th, nb_phi)))
        processed.update(np.where(mask)[0])
    return results


def sr_for_random_frame(data, max_dist, eq=1, seed=RANDOM_SEED):
    """
    Select one random frame from data and compute per-cell Sr.

    :param eq:   1 for Legendre, 2 for Basaran
    :param seed: Random seed for reproducibility
    :return: (result_df, selected_frame_time)
    """
    rng = np.random.default_rng(seed)
    selected_t  = rng.choice(data['POSITION_T'].unique())
    frame       = data[data['POSITION_T'] == selected_t]
    x  = frame['POSITION_X'].values
    y  = frame['POSITION_Y'].values
    th = frame['ELLIPSE_THETA'].values

    fn = sr_per_cell_legendre if eq == 1 else sr_per_cell_basaran
    records = fn(th, x, y, max_dist)
    result_df = pd.DataFrame(records, columns=['X', 'Y', 'Cell_Index', 'Nematic_Order'])
    result_df['Timeframe'] = selected_t
    return result_df, selected_t


# =============================================================================
# 1.7  Varying-distance sweep (Section C.5)
# =============================================================================

def sr_vs_distance(data, dist_values, seed=RANDOM_SEED):
    """
    Compute mean |Sr₁| as a function of neighbourhood radius for one random frame.
    Uses Eq. 1 (Legendre) as the basis for the sweep.

    :param dist_values: Iterable of max_distance values (µm)
    :return: (dict {dist: mean_abs_Sr}, selected_frame_time)
    """
    rng = np.random.default_rng(seed)
    selected_t = rng.choice(data['POSITION_T'].unique())
    frame      = data[data['POSITION_T'] == selected_t]
    x  = frame['POSITION_X'].values
    y  = frame['POSITION_Y'].values
    th = frame['ELLIPSE_THETA'].values

    result = {}
    for d in dist_values:
        records = sr_per_cell_legendre(th, x, y, d)
        if records:
            result[d] = float(np.mean(np.abs([r[3] for r in records])))
        else:
            result[d] = np.nan
    return result, selected_t


print("All helper functions loaded successfully.")


## Section C.1 — Global Sr: All Frames, All Conditions
Computes Sr₁ (Legendre) and Sr₂ (Basaran) for every frame in each condition.
Both equations applied to the entire FOV — no spatial filtering.

Output: `1_Sr_global.csv` — columns: Condition, Timeframe, Sr1_Legendre, Sr2_Basaran, N_cells


In [ ]:
# BUG FIX (03.03.2026): all_results initialised here, not globally.
# In previous notebooks, a single global list was reused across sections,
# causing results to accumulate across re-runs and across sections.
all_results = []

for file_name in file_names:
    data = load_trackmate_csv(os.path.join(DATA_DIR, file_name))
    condition = file_name.split('.')[0]

    for t, group in data.groupby('POSITION_T'):
        x  = group['POSITION_X'].values
        y  = group['POSITION_Y'].values
        th = group['ELLIPSE_THETA'].values
        n  = len(th)
        if n < 2:
            continue

        # Colony centroid for Eq. 2
        cx, cy = np.mean(x), np.mean(y)
        phi    = compute_phi(x, y, cx, cy)

        all_results.append({
            'Condition'   : condition,
            'Timeframe'   : t,
            'Sr1_Legendre': sr_legendre(th),
            'Sr2_Basaran' : sr_basaran(th, phi),
            'N_cells'     : n,
        })

df_global = pd.DataFrame(all_results)
out_path  = os.path.join(OUTPUT_DIR, '1_Sr_global.csv')
df_global.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df_global)} rows)")
print(df_global.groupby('Condition')[['Sr1_Legendre','Sr2_Basaran']].mean().round(3))


## Section C.2 — Quadrant-Based Sr
Divides the 133 × 133 µm FOV into 4 quadrants (66.5 µm each).
Sr is computed within each quadrant using the quadrant geometric centre as the φ reference.
This tests whether alignment is a local micro-domain phenomenon rather than a global colony trait.

Output: `2_Sr_quadrant.csv` — columns: Condition, Timeframe, Quadrant, Sr1_Legendre, Sr2_Basaran, N_cells


In [ ]:
all_results = []

for file_name in file_names:
    data = load_trackmate_csv(os.path.join(DATA_DIR, file_name))
    condition = file_name.split('.')[0]

    for t, group in data.groupby('POSITION_T'):
        for quadrant in ['Q1', 'Q2', 'Q3', 'Q4']:
            sub = filter_by_quadrant(group, quadrant, HALF_FOV)
            if len(sub) < 2:
                continue

            x  = sub['POSITION_X'].values
            y  = sub['POSITION_Y'].values
            th = sub['ELLIPSE_THETA'].values

            # φ relative to fixed quadrant geometric centre
            cx, cy = QUADRANT_CENTRES[quadrant]
            phi    = compute_phi(x, y, cx, cy)

            all_results.append({
                'Condition'   : condition,
                'Timeframe'   : t,
                'Quadrant'    : quadrant,
                'Sr1_Legendre': sr_legendre(th),
                'Sr2_Basaran' : sr_basaran(th, phi),
                'N_cells'     : len(th),
            })

df_quad = pd.DataFrame(all_results)
out_path = os.path.join(OUTPUT_DIR, '2_Sr_quadrant.csv')
df_quad.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df_quad)} rows)")
print(df_quad.groupby(['Condition','Quadrant'])[['Sr1_Legendre','Sr2_Basaran']].mean().round(3))


## Section C.3 — Distance-Based Local Sr (30 µm Neighbourhood)
Cells are grouped into non-overlapping local neighbourhoods (radius = 30 µm).
Sr is computed within each neighbourhood and averaged across the frame.
This avoids the arbitrary FOV subdivision of Section C.2.

The 30 µm threshold was determined empirically by measuring average inter-cell distances in FIJI.

Output: `3_Sr_distance_30um.csv` — columns: Condition, Timeframe, Sr1_Legendre, Sr2_Basaran, N_cells


In [ ]:
all_results = []

for file_name in file_names:
    data = load_trackmate_csv(os.path.join(DATA_DIR, file_name))
    condition = file_name.split('.')[0]
    print(f"Processing {condition}...", end=" ")

    for t, group in data.groupby('POSITION_T'):
        x  = group['POSITION_X'].values
        y  = group['POSITION_Y'].values
        th = group['ELLIPSE_THETA'].values
        if len(th) < 2:
            continue

        all_results.append({
            'Condition'   : condition,
            'Timeframe'   : t,
            'Sr1_Legendre': sr_legendre_local(th, x, y, MAX_DISTANCE),
            'Sr2_Basaran' : sr_basaran_local(th, x, y, MAX_DISTANCE),
            'N_cells'     : len(th),
        })
    print("done")

df_dist = pd.DataFrame(all_results)
out_path = os.path.join(OUTPUT_DIR, '3_Sr_distance_30um.csv')
df_dist.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}  ({len(df_dist)} rows)")
print(df_dist.groupby('Condition')[['Sr1_Legendre','Sr2_Basaran']].mean().round(3))


## Section C.4 — Per-Cell Sr from a Representative Frame
Since Sr is observed to be approximately time-independent (Section C.1), 
a single randomly selected frame per condition is used to generate a spatial map of Sr.
Each point in the output represents one non-overlapping cell neighbourhood (radius = 13 µm ≈ 1/10 FOV).
This enables visualisation of spatial heterogeneity in local coordination.

Output: `4_Sr_per_cell.csv` — columns: Condition, X, Y, Cell_Index, Nematic_Order, Equation, Timeframe


In [ ]:
all_results = []

for file_name in file_names:
    data      = load_trackmate_csv(os.path.join(DATA_DIR, file_name))
    condition = file_name.split('.')[0]

    for eq_label, eq_num in [('Sr1_Legendre', 1), ('Sr2_Basaran', 2)]:
        df_frame, t_sel = sr_for_random_frame(data, MAX_DIST_CELL, eq=eq_num, seed=RANDOM_SEED)
        df_frame['Condition'] = condition
        df_frame['Equation']  = eq_label
        all_results.append(df_frame)
        print(f"{condition} | {eq_label} | frame t={t_sel:.1f}s | {len(df_frame)} cells")

df_percell = pd.concat(all_results, ignore_index=True)
out_path   = os.path.join(OUTPUT_DIR, '4_Sr_per_cell.csv')
df_percell.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}  ({len(df_percell)} rows)")


## Section C.5 — Sr vs Neighbourhood Radius Sweep
Investigates how the neighbourhood radius affects the measured Sr.
Eq. 1 (Legendre) is used for the sweep. A single random frame per condition is selected.
Results reveal the characteristic length scale of collective bacterial alignment.

Output: `5_Sr_vs_distance.csv` — columns: Condition, Max_Distance_um, Mean_Abs_Sr, Timeframe


In [ ]:
all_results = []

for file_name in file_names:
    data      = load_trackmate_csv(os.path.join(DATA_DIR, file_name))
    condition = file_name.split('.')[0]
    print(f"Sweeping distances for {condition}...")

    dist_map, t_sel = sr_vs_distance(data, DIST_SWEEP, seed=RANDOM_SEED)
    for d, sr in dist_map.items():
        all_results.append({
            'Condition'       : condition,
            'Max_Distance_um' : d,
            'Mean_Abs_Sr'     : sr,
            'Timeframe'       : t_sel,
        })

df_sweep = pd.DataFrame(all_results)
out_path  = os.path.join(OUTPUT_DIR, '5_Sr_vs_distance.csv')
df_sweep.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}  ({len(df_sweep)} rows)")


## ⚠️ Equation Labelling Note
The two Sr formulas carry different labels depending on context — this table resolves the naming:

| This notebook | Project report | Formula | Result |
|---|---|---|---|
| `sr_legendre` / `Sr1_Legendre` | Equation 2 | `⟨(3cos²θ − 1)/2⟩` | **Differentiates conditions** — pilG > pilH > WT in dense |
| `sr_basaran` / `Sr2_Basaran` | Equation 3 | `⟨cos(2(θᵢ − φᵢ))⟩` | No significant differences across conditions |

The Legendre formula (Sr₁ here, Eq. 2 in the report) is the **primary result** of this analysis.
The Basaran radial formula (Sr₂ here, Eq. 3 in the report) is retained for completeness and methodological comparison.


## Section C.6 — Visualisation
Three plots:
1. Sr₂ (Basaran) vs time, per condition (grouped by density)
2. Sr₁ vs Sr₂ scatter (global, all conditions) — compares the two formulas
3. Sr vs neighbourhood radius for all 9 conditions


In [ ]:
COLORS     = {'WT': '#2196F3', 'pilH': '#F44336', 'pilGcpdA': '#4CAF50'}
LINESTYLES = {'Dense': '-', 'Medium': '--', 'Dilute': ':'}

def parse_condition(name):
    """Extract strain and density from condition filename stem."""
    strain  = next((s for s in ['pilGcpdA','pilH','WT'] if name.startswith(s)), name)
    density = next((d for d in ['Dense','Medium','Dilute'] if d in name), '')
    return strain, density

# --- Plot 1: Sr₁ (Legendre) vs time per condition ---
# Sr₁ is the formula that differentiated mutants (= Equation 2 in report, Fig 1).
# Sr₂ (Basaran) showed no significant difference (report Fig S20) — see Plot 2 for comparison.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
density_order = ['Dense', 'Medium', 'Dilute']

for ax, density in zip(axes, density_order):
    subset = df_global[df_global['Condition'].str.contains(density)]
    for _, grp in subset.groupby('Condition'):
        strain, den = parse_condition(grp['Condition'].iloc[0])
        grp_sorted  = grp.sort_values('Timeframe')
        ax.plot(grp_sorted['Timeframe'], grp_sorted['Sr1_Legendre'],
                label=strain, color=COLORS.get(strain, 'grey'), linewidth=1.5)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(f'{density} colonies')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Sr₁ (Legendre)')
    ax.legend(fontsize=8)
    ax.set_ylim(-0.5, 1.0)

fig.suptitle('Nematic Order Sr₁ (Legendre) vs Time by Density', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'Fig1_Sr1_vs_time.png'), dpi=200, bbox_inches='tight')
plt.show()

# --- Plot 2: Sr₁ vs Sr₂ comparison (global) ---
fig, ax = plt.subplots(figsize=(6, 5))
for cond, grp in df_global.groupby('Condition'):
    strain, _ = parse_condition(cond)
    ax.scatter(grp['Sr1_Legendre'], grp['Sr2_Basaran'],
               color=COLORS.get(strain, 'grey'), alpha=0.4, s=10, label=strain)
ax.plot([-0.5, 1], [-0.5, 1], 'k--', linewidth=0.8, label='Sr₁ = Sr₂')
ax.set_xlabel('Sr₁ (Legendre)')
ax.set_ylabel('Sr₁ (Legendre)')
ax.set_title('Comparison of Sr Formulas (Global)')
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'Fig2_Sr1_vs_Sr2.png'), dpi=200, bbox_inches='tight')
plt.show()

# --- Plot 3: Sr vs neighbourhood radius ---
fig, ax = plt.subplots(figsize=(8, 5))
for cond, grp in df_sweep.groupby('Condition'):
    strain, density = parse_condition(cond)
    grp_sorted = grp.sort_values('Max_Distance_um')
    ax.plot(grp_sorted['Max_Distance_um'], grp_sorted['Mean_Abs_Sr'],
            color=COLORS.get(strain, 'grey'),
            linestyle=LINESTYLES.get(density, '-'),
            linewidth=1.5, label=cond)
ax.axvline(MAX_DISTANCE, color='orange', linestyle=':', linewidth=1, label=f'{MAX_DISTANCE} µm threshold')
ax.set_xlabel('Neighbourhood Radius (µm)')
ax.set_ylabel('Mean |Sr₁| (Legendre)')
ax.set_title('Nematic Order vs Neighbourhood Radius')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'Fig3_Sr_vs_distance.png'), dpi=200, bbox_inches='tight')
plt.show()

print("All figures saved to:", OUTPUT_DIR)
